In [0]:
from pyspark.sql import SparkSession
import datetime
import re
import os
import time
import pyspark
from pyspark.sql.types import StructType, StructField, BooleanType, DoubleType
from pyspark.sql.types import StringType
from pyspark.sql.types import DateType
from pyspark.sql.types import IntegerType
from pyspark.sql import Row
from datetime import date

In [0]:
spark = (
    SparkSession.builder
    .appName("SCD Type Demo")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")

    # ===== ALL PACKAGES IN ONE PLACE =====
    # .config(
    #     "spark.jars.packages",
    #     ",".join([
    #         "org.apache.spark:spark-token-provider-kafka-0-10_2.12:3.5.6",
    #         "org.postgresql:postgresql:42.7.7"
    #     ])
    # )

    # ===== EXTENSIONS =====
    # .config(
    #     "spark.sql.extensions",
    #     "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
    #     "org.projectnessie.spark.extensions.NessieSparkSessionExtensions"
    # )


    .getOrCreate()
)

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

print("Spark Session Started")

In [0]:
%sql


USE mlb_demo.learning_demo;


CREATE OR REPLACE TABLE  scd_example_one (
    id  BIGINT  PRIMARY KEY,
    name STRING NOT NULL,
    dob DATE NOT NULL
);

## SCD Type: 0

In [0]:
from pyspark.sql.types import LongType

schema_0 = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("dob", DateType(), True),

])

data = [

    (1,'Jordan', date(1978,1,1)),
    (2, 'Joe', date(1979,1,1)),
    (3, 'Mack', date(1979,5,1)),
]

df = spark.createDataFrame(data, schema=schema_0)


df.show()

In [0]:
df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
            .saveAsTable("mlb_demo.learning_demo.scd_example_one")


#### With this nothing is updated or upserted to

## SCD Type: 1

#### This method overwrites old with new data, and therefore does not track historical data.

In [0]:
%sql

USE mlb_demo.learning_demo;

CREATE TABLE IF NOT EXISTS scd_example_scd_typ_1 (

    player_id VARCHAR(10) NOT NULL PRIMARY KEY,
    AB int NOT NULL,
    H INT NOT NULL,
    BB INT NOT NULL,
    SO INT NOT NULL,
    RBI INT NOT NULL
);

In [0]:
schema_1 = StructType([
    StructField("player_id", StringType(), nullable=False),
    StructField("ab", IntegerType(), nullable=False),
    StructField("h", IntegerType(), nullable=False),
    StructField("bb", IntegerType(), nullable=False),
    StructField("so", IntegerType(), nullable=False),
    StructField("rbi", IntegerType(), nullable=False),





])

data = [('60912', 400, 78, 12, 13, 34 ),
        ('60913', 500, 112, 45, 33, 67 ),
        ('60914', 101, 32, 7, 11, 10 ),
        ('60915', 234, 65, 64, 1, 38 )

]


df = spark.createDataFrame(data, schema=schema_1)



df.show()

In [0]:
df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
            .saveAsTable("mlb_demo.learning_demo.scd_example_scd_typ_1")

In [0]:
%sql

SELECT * FROM mlb_demo.learning_demo.scd_example_scd_typ_1


### In SCD Type I, on new updates to the data, the data is updated and the history isn't preserved

#### Unlike, iceberg or delta tables, for postgres this requires a staging table in postgres

In [0]:
new_data = [('60912', 450, 100, 18, 25, 50 ),
        ('60913', 501, 113, 45, 33, 68 ),

]


df = spark.createDataFrame(new_data, schema=schema_1)



df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
            .saveAsTable("mlb_demo.learning_demo.scd_example_scd_typ_1_staging")


In [0]:
%sql


INSERT INTO mlb_demo.learning_demo.scd_example_scd_typ_1 (player_id, AB, H, BB, SO, RBI)
SELECT player_id, AB, H, BB, SO, RBI
FROM mlb_demo.learning_demo.scd_example_scd_typ_1_staging 

ON CONFLICT (player_id)
DO UPDATE SET
  AB  = EXCLUDED.AB,
  H  = EXCLUDED.H,
  BB = EXCLUDED.BB,
  SO = EXCLUDED.SO,
  RBI = EXCLUDED.RBI
;

In [0]:
%sql

MERGE INTO mlb_demo.learning_demo.scd_example_scd_typ_1 target
USING mlb_demo.learning_demo.scd_example_scd_typ_1_staging source
ON target.player_id = source.player_id
WHEN MATCHED THEN
UPDATE SET
  target.AB = source.AB,
  target.H = source.H,
  target.BB = source.BB,
  target.SO = source.SO,
  target.RBI = source.RBI
WHEN NOT MATCHED THEN
INSERT (player_id, AB, H, BB, SO, RBI)
VALUES (source.player_id, source.AB, source.H, source.BB, source.SO, source.RBI);



In [0]:
%sql

SELECT * FROM mlb_demo.learning_demo.scd_example_scd_typ_1


### From here you can see that the values are able to be updated but the historical past values are completely lost

## SCD Type: 2

#### This method tracks historical data by creating multiple records for a given natural key in the dimensional tables with separate surrogate keys and/or different version numbers. Unlimited history is preserved for each insert.

In [0]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()

create_sql = """

DROP TABLE IF EXISTS scd_example_scd_typ_2;

CREATE TABLE IF NOT EXISTS scd_example_scd_typ_2 (

    player_id VARCHAR(10) NOT NULL,
    AB int NOT NULL,
    H INT NOT NULL,
    BB INT NOT NULL,
    SO INT NOT NULL,
    RBI INT NOT NULL,
    ACTIVE BOOLEAN NOT NULL DEFAULT TRUE,
    DATE_REC_CREATED DATE NOT NULL DEFAULT NOW(),
    DATE_REC_ENDED DATE,
    PRIMARY KEY(player_id, ACTIVE)
);
"""

stmt.executeUpdate(create_sql)
stmt.close()
conn.close()


In [0]:
schema_2 = StructType([
    StructField("player_id", StringType(), nullable=False),
    StructField("ab", IntegerType(), nullable=False),
    StructField("h", IntegerType(), nullable=False),
    StructField("bb", IntegerType(), nullable=False),
    StructField("so", IntegerType(), nullable=False),
    StructField("rbi", IntegerType(), nullable=False),
#StructField("active", BooleanType(), nullable=True),
  #  StructField("date_rec_created", DateType(), nullable=True),

])

data = [

    ('60912', 400, 78, 12, 13, 34 ),
    ('60913', 500, 112, 45, 33, 67 ),
    ('60914', 101, 32, 7, 11, 10  ),
    ('60915', 234, 65, 64, 1, 38  )

]




df = spark.createDataFrame(data, schema=schema_2)


df.show()

In [0]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_2") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [0]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_2") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


#### Notice here there is a true and date rec created fields to let you know what is the most recent version of their record, and the data this record was inserted

### Since this is postgres staging table again

In [0]:
new_data = [('60912', 450, 100, 18, 25, 50 ),
        ('60913', 501, 113, 45, 33, 68 ),
        ('70913', 123, 61, 1, 4,   42 ), # NEW RECORD FOR PLAYER ID


]


df = spark.createDataFrame(new_data, schema=schema_2)



df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_2_staging") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()


### MERGE Operation, also with setting the less recent record to not active

In [0]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()


upsert_sql = """
MERGE INTO scd_example_scd_typ_2 AS target
USING (
    SELECT
        player_id,
        AB, H, BB, SO, RBI
    FROM scd_example_scd_typ_2_staging
) AS source
ON target.player_id = source.player_id
   AND target.active = TRUE
WHEN MATCHED AND (
       target.AB  <> source.AB
    OR target.H   <> source.H
    OR target.BB  <> source.BB
    OR target.SO  <> source.SO
    OR target.RBI <> source.RBI
) THEN
    -- expire old row
    UPDATE SET
        active = FALSE,
        DATE_REC_ENDED = NOW()

WHEN NOT MATCHED THEN
    -- insert new version
    INSERT (
        player_id,
        AB, H, BB, SO, RBI
    )
    VALUES (
        source.player_id,
        source.AB, source.H, source.BB, source.SO, source.RBI
    );


INSERT INTO scd_example_scd_typ_2 (
    player_id,
    AB, H, BB, SO, RBI
)
SELECT
    s.player_id,
    s.AB, s.H, s.BB, s.SO, s.RBI
FROM scd_example_scd_typ_2_staging s
LEFT JOIN scd_example_scd_typ_2 t
  ON s.player_id = t.player_id
 AND t.active = TRUE
WHERE t.player_id IS NULL;

;
"""

stmt.executeUpdate(upsert_sql)
stmt.close()
conn.close()


In [0]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_2") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


#### As you can see from here, the newer records are indicated from a true value in the active column and a null date_rec_ended, this allows for you to see the historical values of a record while continually updating

# SCD: Type 3

#### This method tracks changes using separate columns and preserves limited history. The Type 3 preserves limited history as it is limited to the number of columns designated for storing historical data. The original table structure in Type 1 and Type 2 is the same but Type 3 adds additional columns. In the following example, an additional column has been added to the table to record the supplier's original state - only the previous history is stored.

In [0]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()

create_sql = """

DROP TABLE IF EXISTS scd_example_scd_typ_3;

CREATE TABLE IF NOT EXISTS scd_example_scd_typ_3 (

    player_id VARCHAR(10) NOT NULL,
    BA NUMERIC(4,3) NOT NULL,
    MOST_RECENT_BA NUMERIC(4,3) ,
    DATE_REC_EFF DATE NOT NULL DEFAULT NOW(),
    PRIMARY KEY(player_id)
);
"""

stmt.executeUpdate(create_sql)
stmt.close()
conn.close()


In [0]:


schema_3 = StructType([

    StructField("player_id", StringType(), False),
    StructField("ba", DoubleType(), False)
])



data = [

    ('67182', .234),
    ('67183', .308),
    ('67184', .412),
    ('67185', .112),
]


df = spark.createDataFrame(data, schema=schema_3)

df.show()

In [0]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_3") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [0]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_3") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


## Staging Table and this will be similar to an upsert

In [0]:
new_data = [(
    '67182', .278
),
    ('67183', .423),

    ('98192', .342)
]


df = spark.createDataFrame(new_data, schema=schema_3)

df.show()

In [0]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_3_staging") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [0]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()


upsert_sql = """
MERGE INTO scd_example_scd_typ_3 AS target
USING scd_example_scd_typ_3_staging AS source
ON target.player_id = source.player_id

WHEN MATCHED THEN
UPDATE SET
    MOST_RECENT_BA = target.BA,
    DATE_REC_EFF   = NOW(),
    BA             = source.BA


WHEN NOT MATCHED THEN
INSERT (
    player_id,
    BA,
    MOST_RECENT_BA,
    DATE_REC_EFF
)
VALUES (
    source.player_id,
    source.BA,
    NULL,
    NOW()
);


"""

stmt.executeUpdate(upsert_sql)
stmt.close()
conn.close()


In [0]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_3") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


#### Now this preserves partial history through the most recent column and it lets you know the last date the data has been updated

# SCD Type: 4

#### The Type 4 method is usually referred to as using "history tables", where one table keeps the current data, and an additional table is used to keep a record of some or all changes. Both the surrogate keys are referenced in the fact table to enhance query performance.

In [0]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()

create_sql = """

DROP TABLE IF EXISTS scd_example_scd_typ_4;

CREATE TABLE IF NOT EXISTS scd_example_scd_typ_4 (

    player_sg_key SERIAL NOT NULL PRIMARY KEY,
    player_id VARCHAR(10) NOT NULL,
    AB int NOT NULL,
    H INT NOT NULL,
    BB INT NOT NULL,
    SO INT NOT NULL,
    RBI INT NOT NULL,
    UNIQUE(player_id)
);

DROP TABLE IF EXISTS scd_history_table;
CREATE TABLE IF NOT EXISTS scd_history_table (

    player_sg_key SERIAL NOT NULL PRIMARY KEY,
    player_id VARCHAR(10) NOT NULL,
    AB int NOT NULL,
    H INT NOT NULL,
    BB INT NOT NULL,
    SO INT NOT NULL,
    RBI INT NOT NULL,
    DATE_REC_CREATED DATE NOT NULL DEFAULT NOW()



)


"""

stmt.executeUpdate(create_sql)
stmt.close()
conn.close()


#### Notice the Unique index here to not allow for it to have multiple player_ids in the table while not having it as the primary key

In [0]:
schema_4 = StructType([
    StructField("player_id", StringType(), nullable=False),
    StructField("ab", IntegerType(), nullable=False),
    StructField("h", IntegerType(), nullable=False),
    StructField("bb", IntegerType(), nullable=False),
    StructField("so", IntegerType(), nullable=False),
    StructField("rbi", IntegerType(), nullable=False),
#StructField("active", BooleanType(), nullable=True),
  #  StructField("date_rec_created", DateType(), nullable=True),

])

data = [

    ('60912', 400, 78, 12, 13, 34 ),
    ('60913', 500, 112, 45, 33, 67 ),
    ('60914', 101, 32, 7, 11, 10  ),
    ('60915', 234, 65, 64, 1, 38  )

]




df = spark.createDataFrame(data, schema=schema_4)


df.show()

In [0]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_4") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()


df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_history_table") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [0]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_4") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


In [0]:
new_data = [

    ('60914', 102, 33, 7, 11, 10),
    ('60913', 502, 113, 46, 33, 68),
    ('79199', 345, 79, 11, 87, 45)
]


df = spark.createDataFrame(new_data, schema=schema_4)

df.show()

In [0]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_4_staging") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [0]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()

upsert_sql = """
MERGE INTO scd_history_table AS target
USING scd_example_scd_typ_4_staging AS source
ON target.player_id = source.player_id and
       target.AB  = source.AB
    AND target.H  = source.H
    AND target.BB  = source.BB
    AND target.SO  = source.SO
    AND target.RBI <> source.RBI
WHEN MATCHED THEN
    DO NOTHING


WHEN NOT MATCHED
  THEN
    INSERT (
        player_id,
        AB,
        H,
        BB,
        SO,
        RBI
    )
    VALUES (
        source.player_id,
        source.AB,
        SOURCE.H,
        SOURCE.BB,
        SOURCE.SO,
        SOURCE.RBI
    );

-- DELETE ROWS FROM TYP TBL WITH ID

DELETE FROM scd_example_scd_typ_4 t
USING scd_example_scd_typ_4_staging s
WHERE t.player_id = s.player_id;

INSERT INTO scd_example_scd_typ_4(
        player_id,
        AB,
        H,
        BB,
        SO,
        RBI
    )
SELECT player_id, AB, H, BB, SO, RBI
FROM scd_example_scd_typ_4_staging;



"""

stmt.executeUpdate(upsert_sql)
stmt.close()
conn.close()


In [0]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_4") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


In [0]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_history_table") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


#### It's not perfect implementation, but this shows both the history table and the dimension table above which would show the most recent form of the data

##### Just to also wrap up, I wanted to note that the SQL here is not fully bulletproof so for the SQL demonstrated here it'll prob be better to run them inside of SQL functions or stored procedures for making the actions more concurrent and and failure proof!!

In [0]:
spark.stop()